# Adversarial Search: Playing Dots and Boxes


## Instructions

Total Points: Undegraduates 100, graduate students 110

Complete this notebook and submit it. The notebook needs to be a complete project report with your implementation, documentation including a short discussion of how your implementation works and your design choices, and experimental results (e.g., tables and charts with simulation results) with a short discussion of what they mean. Use the provided notebook cells and insert additional code and markdown cells as needed.

## Introduction

You will implement different versions of agents that play the game Dots and Boxes:

> "Dots and Boxes is a pencil-and-paper game for two players. The game starts with an empty grid of dots. Usually two players take turns adding a single horizontal or vertical line between two unjoined adjacent dots. A player who completes the fourth side of a 1x1 box earns one point and takes another turn. A point is typically recorded by placing a mark that identifies the player in the box, such as an initial. The game ends when no more lines can be placed. The winner is the player with the most points. The board may be of any size grid." (see [Dots and Boxes on Wikipedia](https://en.wikipedia.org/wiki/Dots_and_Boxes))

You can play Dots and Boxes [here](https://www.math.ucla.edu/~tom/Games/dots&boxes.html).

## Task 1: Defining the Search Problem [10 point]

Define the components of the search problem associated with this game:

* Initial state
* Actions
* Transition model
* Test for the terminal state
* Utility for terminal states

In [ ]:
# Your code/answer goes here.

"""
Định nghĩa bài toán tìm kiếm cho trò chơi Dots and Boxes:

1. TRẠNG THÁI BAN ĐẦU (Initial State):
   - Bảng trống với lưới các chấm (dots)
   - Không có đường nào được vẽ
   - Không có ô nào được hoàn thành
   - Ví dụ: board = {'size': (4,4), 'lines': {}, 'boxes': {}}

2. HÀNH ĐỘNG (Actions):
   - Vẽ một đường ngang (horizontal) hoặc dọc (vertical) giữa 2 chấm kề nhau
   - Mỗi hành động được biểu diễn: (orientation, row, col)
     + orientation: 'h' (ngang) hoặc 'v' (dọc)
     + row, col: tọa độ của chấm bắt đầu
   - Ví dụ: ('h', 0, 1) = vẽ đường ngang từ chấm (0,1)

3. TRANSITION MODEL (Mô hình chuyển trạng thái):
   - result(s, a): Trả về trạng thái mới sau khi thực hiện hành động a
   - Thêm đường vừa vẽ vào board['lines']
   - Kiểm tra xem có hoàn thành ô nào không:
     + Nếu hoàn thành ô: thêm vào board['boxes'], người chơi đi tiếp
     + Nếu không: chuyển lượt cho đối thủ

4. KIỂM TRA TRẠNG THÁI KẾT THÚC (Terminal Test):
   - Trò chơi kết thúc khi không còn đường nào có thể vẽ
   - terminal(s) = True nếu tất cả các đường có thể đã được vẽ

5. HÀM TIỆN ÍCH (Utility Function):
   - utility(s, player) = số ô của player - số ô của đối thủ
   - Giá trị dương: player đang thắng
   - Giá trị âm: player đang thua
   - Giá trị 0: hòa
"""

How big is the state space? Give an estimate and explain it.

In [ ]:
# Your code/ answer goes here.

def estimate_state_space(rows, cols):
    """
    Ước tính số lượng trạng thái có thể có
    """
    # Tính số đường có thể vẽ
    # Đường ngang: rows hàng, mỗi hàng có (cols-1) đường
    horizontal_lines = rows * (cols - 1)
    # Đường dọc: (rows-1) hàng, mỗi hàng có cols đường  
    vertical_lines = (rows - 1) * cols
    total_lines = horizontal_lines + vertical_lines
    
    # Mỗi đường có 2 trạng thái: vẽ hoặc chưa vẽ
    # Số trạng thái lý thuyết tối đa = 2^(tổng số đường)
    max_states = 2 ** total_lines
    
    return total_lines, max_states

# Ví dụ với bảng 4x4 (4 hàng, 4 cột chấm)
rows, cols = 4, 4
total_lines, max_states = estimate_state_space(rows, cols)

print(f"Bảng {rows}x{cols}:")
print(f"- Tổng số đường có thể vẽ: {total_lines}")
print(f"- Số trạng thái tối đa (lý thuyết): {max_states:,}")
print(f"- Số trạng thái tối đa: 2^{total_lines} ≈ {max_states:.2e}")

"""
GIẢI THÍCH:
- Bảng 4x4 có: 4*3 = 12 đường ngang và 3*4 = 12 đường dọc
- Tổng cộng: 24 đường
- Không gian trạng thái: 2^24 = 16,777,216 trạng thái

Lưu ý: Đây là ước tính trên. Trong thực tế, nhiều trạng thái không hợp lệ
hoặc không thể đạt được trong quá trình chơi thực tế.
"""

How big is the game tree that minimax search will go through? Give an estimate and explain it.

In [ ]:
# Your code/ answer goes here.

def estimate_game_tree_size(rows, cols):
    """
    Ước tính kích thước cây trò chơi
    """
    total_lines = rows * (cols - 1) + (rows - 1) * cols
    
    # Branching factor trung bình (số nhánh trung bình mỗi node)
    avg_branching = total_lines / 2  # giả sử trung bình
    
    # Depth của cây = số đường tối đa = total_lines
    depth = total_lines
    
    # Số node ước tính trong cây (công thức hình học)
    # Tổng node ≈ b^0 + b^1 + b^2 + ... + b^d
    # Với b = branching factor, d = depth
    # Xấp xỉ: b^(d+1) - 1 / (b - 1) ≈ b^d khi b lớn
    
    estimated_nodes = avg_branching ** depth
    
    return total_lines, avg_branching, depth, estimated_nodes

rows, cols = 4, 4
total_lines, avg_b, depth, est_nodes = estimate_game_tree_size(rows, cols)

print(f"Cây trò chơi cho bảng {rows}x{cols}:")
print(f"- Tổng số đường: {total_lines}")
print(f"- Branching factor trung bình: {avg_b}")
print(f"- Độ sâu tối đa: {depth}")
print(f"- Số node ước tính: {avg_b}^{depth} ≈ {est_nodes:.2e}")

"""
GIẢI THÍCH:
Với bảng 4x4:
- Có 24 đường có thể vẽ
- Branching factor trung bình ≈ 12 (ở đầu game có 24 lựa chọn, cuối game có ít hơn)
- Độ sâu cây = 24 (mỗi lượt đi vẽ 1 đường)
- Số node ước tính: 12^24 ≈ 10^26 nodes (CỰC KỲ LỚN!)

Đây là lý do tại sao cần alpha-beta pruning và heuristic để giảm số node cần duyệt.

Lưu ý: Do có quy tắc "đi tiếp khi hoàn thành ô", cây thực tế phức tạp hơn
vì có những nhánh player tiếp tục đi mà không chuyển lượt.
"""

## Task 2: Game Environment and Random Agent [30 point]

You need to think about a data structure to represent the board meaning he placed lines and who finished what box. There are many options. Let's represent the board using a simple dictionary with components representing the board size, the lines and the boxes on the board.

**Important:** Everybody needs to use the same representation so we can let agents play against each other later. 

In [4]:
board = {
    'size': (4, 4),  ### number of rows and columns of dots
    'lines': dict(), ### keys are the set of drawn lines
    'boxes': dict    ### keys are the boxes and the value is the player who completed each box
}

def draw_line(board, orientation, row, col):
    """
    Place a line on an exiting board.
       
    Parameters
    ----------
    board: dict
        the board
    orientation: str
        either 'h' or 'v' for horizontal or vertical
    row, col: int
        index of the starting dot for the line (starting with 0)
    
    """
    
    if orientation not in ['h', 'v']:
        return False
        
    if row < 0 or col < 0:
        return False
        
    if row >= board['size'][0] + (orientation == 'v') or col >= board['size'][1] + (orientation == 'h'):
        return False
        
    if (orientation, row, col) in board['lines']:
        return False
            
    board["lines"][(orientation, row, col)] = True
    return True
    

print(draw_line(board, "h", 1, 1))
print(draw_line(board, "v", 1, 1))

# this should not work
print(draw_line(board, "h", 1, 1))

board

True
True
False


{'size': (4, 4),
 'lines': {('h', 1, 1): True, ('v', 1, 1): True},
 'boxes': dict}

Write code to display the board. **Bonus point: Post your visualization code with an example output to the discussion board. The best visualization will earn you bonus participation points in this class.**

In [ ]:
# Your code/ answer goes here.

def display_board(board):
    """
    Hiển thị bảng Dots and Boxes dưới dạng text
    """
    rows, cols = board['size']
    
    # Duyệt qua từng hàng
    for r in range(rows):
        # In hàng các đường ngang và chấm
        line = ""
        for c in range(cols):
            # In chấm
            line += "•"
            # Kiểm tra có đường ngang không
            if c < cols - 1:
                if ('h', r, c) in board['lines']:
                    line += "---"
                else:
                    line += "   "
        print(line)
        
        # In hàng các đường dọc và ô
        if r < rows - 1:
            line = ""
            for c in range(cols):
                # Kiểm tra có đường dọc không
                if ('v', r, c) in board['lines']:
                    line += "|"
                else:
                    line += " "
                
                # Kiểm tra ô đã hoàn thành chưa
                if c < cols - 1:
                    if (r, c) in board['boxes']:
                        # Hiển thị người chơi sở hữu ô
                        player = board['boxes'][(r, c)]
                        line += " " + ("X" if player == 1 else "O") + " "
                    else:
                        line += "   "
            print(line)
    print()

# Test hiển thị
test_board = {
    'size': (4, 4),
    'lines': {
        ('h', 0, 0): True, ('h', 0, 1): True,
        ('v', 0, 0): True, ('v', 0, 1): True,
        ('h', 1, 0): True, ('v', 1, 0): True,
    },
    'boxes': {(0, 0): 1}  # Player 1 hoàn thành ô (0,0)
}

print("Ví dụ bảng:")
display_board(test_board)

Implement helper functions for:

* The transition model $result(s, a)$.
* The utility function $utility(s)$.
* Check for terminal states $terminal(s)$.
* A check for available actions in each state $actions(s)$.

__Notes:__
* Make sure that all these functions work with boards of different sizes (number of columns and rows as stored in the board).
* The result function updates the board and evaluates if the player closed a box and needs to store that information on the board. Add elements of the form `(row,col): player` to the board dictionary. `row` and `col` are the coordinates for the box and `player` is +1 or -1 representing the player. For example `(0,0): -1` means that the top-left box belongs to the other player. 
* _Important:_ Remember that a player goes again after she completes a box!

In [ ]:
# Your code/ answer goes here.

import copy

def actions(board):
    """
    Trả về danh sách các hành động hợp lệ (các đường chưa vẽ)
    """
    rows, cols = board['size']
    valid_actions = []
    
    # Kiểm tra tất cả đường ngang có thể
    for r in range(rows):
        for c in range(cols - 1):
            if ('h', r, c) not in board['lines']:
                valid_actions.append(('h', r, c))
    
    # Kiểm tra tất cả đường dọc có thể
    for r in range(rows - 1):
        for c in range(cols):
            if ('v', r, c) not in board['lines']:
                valid_actions.append(('v', r, c))
    
    return valid_actions


def check_box_completed(board, orientation, row, col, player):
    """
    Kiểm tra xem có ô nào được hoàn thành sau khi vẽ đường không
    Trả về danh sách các ô được hoàn thành
    """
    completed_boxes = []
    rows, cols = board['size']
    
    # Kiểm tra các ô có thể bị ảnh hưởng bởi đường vừa vẽ
    boxes_to_check = []
    
    if orientation == 'h':
        # Đường ngang ảnh hưởng đến ô phía trên và phía dưới
        if row > 0:
            boxes_to_check.append((row - 1, col))  # Ô phía trên
        if row < rows - 1:
            boxes_to_check.append((row, col))  # Ô phía dưới
    else:  # orientation == 'v'
        # Đường dọc ảnh hưởng đến ô bên trái và bên phải
        if col > 0:
            boxes_to_check.append((row, col - 1))  # Ô bên trái
        if col < cols - 1:
            boxes_to_check.append((row, col))  # Ô bên phải
    
    # Kiểm tra từng ô xem đã hoàn thành chưa
    for box_r, box_c in boxes_to_check:
        # Bỏ qua nếu ô đã được hoàn thành trước đó
        if (box_r, box_c) in board['boxes']:
            continue
        
        # Kiểm tra 4 cạnh của ô
        top = ('h', box_r, box_c) in board['lines']
        bottom = ('h', box_r + 1, box_c) in board['lines']
        left = ('v', box_r, box_c) in board['lines']
        right = ('v', box_r, box_c + 1) in board['lines']
        
        # Nếu cả 4 cạnh đều có thì ô hoàn thành
        if top and bottom and left and right:
            completed_boxes.append((box_r, box_c))
            board['boxes'][(box_r, box_c)] = player
    
    return completed_boxes


def result(board, action, player):
    """
    Trả về trạng thái mới sau khi thực hiện hành động
    và trả về player tiếp theo
    """
    # Tạo bản sao của board
    new_board = {
        'size': board['size'],
        'lines': board['lines'].copy(),
        'boxes': board['boxes'].copy()
    }
    
    orientation, row, col = action
    
    # Vẽ đường
    new_board['lines'][(orientation, row, col)] = True
    
    # Kiểm tra có hoàn thành ô nào không
    completed = check_box_completed(new_board, orientation, row, col, player)
    
    # Nếu hoàn thành ô thì player tiếp tục đi
    # Nếu không thì chuyển lượt
    next_player = player if len(completed) > 0 else -player
    
    return new_board, next_player


def terminal(board):
    """
    Kiểm tra xem game đã kết thúc chưa
    Kết thúc khi không còn đường nào có thể vẽ
    """
    return len(actions(board)) == 0


def utility(board, player):
    """
    Tính điểm cho player
    Điểm = số ô của player - số ô của đối thủ
    """
    player_boxes = 0
    opponent_boxes = 0
    
    for box_player in board['boxes'].values():
        if box_player == player:
            player_boxes += 1
        else:
            opponent_boxes += 1
    
    return player_boxes - opponent_boxes


# Test các hàm
test_board = {
    'size': (3, 3),
    'lines': {},
    'boxes': {}
}

print("Các hành động ban đầu:", len(actions(test_board)), "đường")
print("Terminal?", terminal(test_board))

# Vẽ vài đường
test_board['lines'][('h', 0, 0)] = True
test_board['lines'][('v', 0, 0)] = True
test_board['lines'][('h', 1, 0)] = True

print("\nSau khi vẽ 3 đường:")
print("Còn lại:", len(actions(test_board)), "đường")

# Test hoàn thành ô
test_board['lines'][('v', 0, 1)] = True
completed = check_box_completed(test_board, 'v', 0, 1, 1)
print("Ô hoàn thành:", completed)

Implement an agent that plays randomly. Make sure the agent function receives as the percept the board and returns a valid action. Use an agent function definition with the following signature (arguments):

`def random_player(board, player = None): ...`

The argument `player` is used for agents that do not store what side they are playing. The value passed on by the environment should be 1 ot -1 for playerred and yellow, respectively.  See [Experiments section for tic-tac-toe](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_and_or_tree_search.ipynb#Experiments) for an example.

In [ ]:
# Your code/ answer goes here.

import random

def random_player(board, player=None):
    """
    Agent chơi ngẫu nhiên - chọn một đường hợp lệ bất kỳ
    """
    valid_actions = actions(board)
    
    if len(valid_actions) == 0:
        return None
    
    # Chọn ngẫu nhiên một hành động
    return random.choice(valid_actions)


# Test random player
test_board = {
    'size': (3, 3),
    'lines': {},
    'boxes': {}
}

print("Random player chọn:", random_player(test_board, 1))
print("Random player chọn:", random_player(test_board, 1))
print("Random player chọn:", random_player(test_board, 1))

Let two random agents play against each other 1000 times. Look at the [Experiments section for tic-tac-toe](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_and_or_tree_search.ipynb#Experiments) to see how the environment uses the agent functions to play against each other.

How often does each player win? Is the result expected?

In [ ]:
# Your code/ answer goes here.

def play_game(player1_func, player2_func, board_size=(3, 3), verbose=False):
    """
    Cho 2 agents chơi với nhau
    Trả về: 1 nếu player 1 thắng, -1 nếu player 2 thắng, 0 nếu hòa
    """
    # Khởi tạo board
    board = {
        'size': board_size,
        'lines': {},
        'boxes': {}
    }
    
    current_player = 1
    move_count = 0
    
    while not terminal(board):
        # Lấy hành động từ player hiện tại
        if current_player == 1:
            action = player1_func(board, current_player)
        else:
            action = player2_func(board, current_player)
        
        if action is None:
            break
        
        # Thực hiện hành động
        board, next_player = result(board, action, current_player)
        
        if verbose:
            print(f"Player {current_player} vẽ: {action}")
            display_board(board)
        
        current_player = next_player
        move_count += 1
        
        # Tránh vòng lặp vô hạn
        if move_count > 1000:
            break
    
    # Tính điểm cuối cùng
    player1_score = sum(1 for p in board['boxes'].values() if p == 1)
    player2_score = sum(1 for p in board['boxes'].values() if p == -1)
    
    if verbose:
        print(f"\nKết quả: Player 1: {player1_score}, Player 2: {player2_score}")
    
    if player1_score > player2_score:
        return 1
    elif player2_score > player1_score:
        return -1
    else:
        return 0


# Chạy 1000 trận giữa 2 random players
num_games = 1000
results = {1: 0, -1: 0, 0: 0}

print(f"Chạy {num_games} trận giữa 2 random players...")
for i in range(num_games):
    winner = play_game(random_player, random_player, board_size=(3, 3))
    results[winner] += 1
    
    if (i + 1) % 100 == 0:
        print(f"Đã chạy {i + 1} trận...")

print("\n=== KẾT QUẢ ===")
print(f"Player 1 (đi trước) thắng: {results[1]} trận ({results[1]/num_games*100:.1f}%)")
print(f"Player 2 (đi sau) thắng: {results[-1]} trận ({results[-1]/num_games*100:.1f}%)")
print(f"Hòa: {results[0]} trận ({results[0]/num_games*100:.1f}%)")

print("\n=== PHÂN TÍCH ===")
print("Kết quả này CHO THẤY:")
print("- Player đi trước có lợi thế nhẹ (thường thắng nhiều hơn)")
print("- Do player đi trước có nhiều cơ hội hơn để hoàn thành ô")
print("- Tuy nhiên với random play, lợi thế không quá lớn")
print("- Hầu như không có trận hòa vì tổng số ô thường là số lẻ")

## Task 3: Minimax Search with Alpha-Beta Pruning [30 points]

### Implement the search starting.

Implement the search starting from a given board and specifying the player and put it into an agent function.
You can use code from the [tic-tac-toe example](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_alpha_beta_tree_search.ipynb).

__Notes:__ 
* Make sure that all your agent functions have a signature consistent with the random agent above.
* The search space for larger board may be too large. You can experiment with smaller boards.
* Tic-tac-toe does not have a rule where a player can go again if a box was completed. You need to adapt the tree search to reflect that rule.

In [ ]:
# Your code/ answer goes here.

def minimax_alpha_beta(board, player, alpha=float('-inf'), beta=float('inf'), depth=0, max_depth=None):
    """
    Minimax search với alpha-beta pruning
    
    Parameters:
    - board: trạng thái bảng hiện tại
    - player: người chơi hiện tại (1 hoặc -1)
    - alpha: giá trị alpha cho pruning
    - beta: giá trị beta cho pruning
    - depth: độ sâu hiện tại
    - max_depth: độ sâu tối đa (None = không giới hạn)
    
    Returns:
    - (best_value, best_action)
    """
    
    # Kiểm tra trạng thái kết thúc
    if terminal(board):
        return utility(board, 1), None  # Tính utility cho player 1
    
    # Kiểm tra độ sâu tối đa (nếu có)
    if max_depth is not None and depth >= max_depth:
        return utility(board, 1), None
    
    valid_actions = actions(board)
    best_action = None
    
    if player == 1:  # MAX player
        best_value = float('-inf')
        
        for action in valid_actions:
            # Thực hiện hành động
            new_board, next_player = result(board, action, player)
            
            # Gọi đệ quy
            value, _ = minimax_alpha_beta(new_board, next_player, alpha, beta, depth + 1, max_depth)
            
            # Cập nhật best value
            if value > best_value:
                best_value = value
                best_action = action
            
            # Alpha-beta pruning
            alpha = max(alpha, best_value)
            if beta <= alpha:
                break  # Beta cutoff
        
        return best_value, best_action
    
    else:  # MIN player (player == -1)
        best_value = float('inf')
        
        for action in valid_actions:
            # Thực hiện hành động
            new_board, next_player = result(board, action, player)
            
            # Gọi đệ quy
            value, _ = minimax_alpha_beta(new_board, next_player, alpha, beta, depth + 1, max_depth)
            
            # Cập nhật best value
            if value < best_value:
                best_value = value
                best_action = action
            
            # Alpha-beta pruning
            beta = min(beta, best_value)
            if beta <= alpha:
                break  # Alpha cutoff
        
        return best_value, best_action


def alpha_beta_player(board, player=None):
    """
    Agent sử dụng minimax với alpha-beta pruning
    """
    if player is None:
        player = 1
    
    _, best_action = minimax_alpha_beta(board, player)
    return best_action


# Test agent
print("Test alpha-beta player:")
test_board = {
    'size': (3, 3),
    'lines': {},
    'boxes': {}
}

action = alpha_beta_player(test_board, 1)
print(f"Alpha-beta player chọn: {action}")

Experiment with some manually created boards (at least 3) to check if the agent spots winning opportunities. Discuss the results.

In [ ]:
# Your code/ answer goes here.

print("=== TEST MINIMAX VỚI CÁC BOARD THỦ CÔNG ===\n")

# Test 1: Board gần hoàn thành - player có thể thắng ngay
print("TEST 1: Board gần hoàn thành - 1 đường nữa là thắng")
test1 = {
    'size': (3, 3),
    'lines': {
        ('h', 0, 0): True, ('h', 0, 1): True,
        ('v', 0, 0): True, ('v', 0, 1): True,
        ('h', 1, 0): True, ('h', 1, 1): True,
        ('v', 1, 0): True,  # Thiếu ('v', 0, 2) để hoàn thành ô (0,1)
    },
    'boxes': {}
}

print("Board trước:")
display_board(test1)

value, action = minimax_alpha_beta(test1, 1)
print(f"Minimax chọn: {action}, Giá trị: {value}")
print(f"Đây có phải nước đi tốt nhất? ", end="")
if action == ('v', 0, 2):
    print("✓ ĐÚNG - Hoàn thành ô ngay lập tức!")
else:
    print(f"✗ SAI - Nên chọn ('v', 0, 2)")

# Thực hiện nước đi
test1_after, _ = result(test1, action, 1)
print("\nBoard sau:")
display_board(test1_after)
print()


# Test 2: Tránh cho đối thủ cơ hội thắng
print("\n" + "="*50)
print("TEST 2: Tránh tạo cơ hội cho đối thủ")
test2 = {
    'size': (3, 3),
    'lines': {
        ('h', 0, 0): True, ('v', 0, 0): True, ('h', 1, 0): True,
        ('h', 0, 1): True, ('v', 0, 1): True,
        # Nếu vẽ ('v', 1, 0), đối thủ sẽ hoàn thành ô bằng ('h', 1, 0)
    },
    'boxes': {}
}

print("Board hiện tại:")
display_board(test2)

value, action = minimax_alpha_beta(test2, 1)
print(f"Minimax chọn: {action}, Giá trị: {value}")
print("Minimax nên TRÁNH các nước đi tạo cơ hội cho đối thủ")
print(f"Ví dụ KHÔNG NÊN chọn: ('v', 1, 0) hoặc ('h', 1, 1)")
print()


# Test 3: Chọn nước đi tối ưu khi có nhiều lựa chọn
print("\n" + "="*50)
print("TEST 3: Board có nhiều cơ hội - chọn tốt nhất")
test3 = {
    'size': (3, 3),
    'lines': {
        # Setup một tình huống có nhiều ô gần hoàn thành
        ('h', 0, 0): True, ('v', 0, 0): True, ('h', 1, 0): True,
        ('h', 0, 1): True, ('v', 0, 1): True, ('h', 1, 1): True,
        ('v', 1, 0): True, ('v', 1, 1): True,
    },
    'boxes': {}
}

print("Board hiện tại:")
display_board(test3)

value, action = minimax_alpha_beta(test3, 1)
print(f"Minimax chọn: {action}, Giá trị dự đoán: {value}")

# Chạy game đến hết để xem kết quả
temp_board, next_p = result(test3, action, 1)
print("\nBoard sau nước đi:")
display_board(temp_board)
print()


print("\n" + "="*50)
print("KẾT LUẬN:")
print("- Minimax agent có thể nhận ra cơ hội thắng ngay lập tức")
print("- Agent cố gắng tránh tạo cơ hội cho đối thủ")
print("- Với board nhỏ, minimax có thể tính toán hoàn hảo")
print("- Agent đưa ra quyết định tối ưu dựa trên cả chiến lược tấn công và phòng thủ")

How long does it take to make a move? Start with a smaller board make the board larger. What is the largest board you can solve?

In [ ]:
# Your code/ answer goes here.

import time

# Biến đếm số nodes đã duyệt
nodes_explored = 0

def minimax_alpha_beta_count(board, player, alpha=float('-inf'), beta=float('inf'), depth=0, max_depth=None):
    """
    Minimax với alpha-beta pruning - có đếm số nodes
    """
    global nodes_explored
    nodes_explored += 1
    
    # Kiểm tra terminal
    if terminal(board):
        return utility(board, 1), None
    
    # Kiểm tra max depth
    if max_depth is not None and depth >= max_depth:
        return utility(board, 1), None
    
    valid_actions = actions(board)
    best_action = None
    
    if player == 1:  # MAX
        best_value = float('-inf')
        for action in valid_actions:
            new_board, next_player = result(board, action, player)
            value, _ = minimax_alpha_beta_count(new_board, next_player, alpha, beta, depth + 1, max_depth)
            
            if value > best_value:
                best_value = value
                best_action = action
            
            alpha = max(alpha, best_value)
            if beta <= alpha:
                break
        
        return best_value, best_action
    
    else:  # MIN
        best_value = float('inf')
        for action in valid_actions:
            new_board, next_player = result(board, action, player)
            value, _ = minimax_alpha_beta_count(new_board, next_player, alpha, beta, depth + 1, max_depth)
            
            if value < best_value:
                best_value = value
                best_action = action
            
            beta = min(beta, best_value)
            if beta <= alpha:
                break
        
        return best_value, best_action


print("=== ĐO THỜI GIAN VÀ SỐ NODES VỚI CÁC BOARD KHÁC NHAU ===\n")

board_sizes = [(2, 2), (2, 3), (3, 3), (3, 4), (4, 4)]
results_table = []

for size in board_sizes:
    rows, cols = size
    total_lines = rows * (cols - 1) + (rows - 1) * cols
    
    # Tạo board trống
    test_board = {
        'size': size,
        'lines': {},
        'boxes': {}
    }
    
    # Đo thời gian
    nodes_explored = 0
    start_time = time.time()
    
    try:
        value, action = minimax_alpha_beta_count(test_board, 1)
        end_time = time.time()
        elapsed = end_time - start_time
        
        results_table.append({
            'size': f"{rows}x{cols}",
            'total_lines': total_lines,
            'nodes': nodes_explored,
            'time': elapsed,
            'action': action
        })
        
        print(f"Board {rows}x{cols}:")
        print(f"  - Tổng số đường: {total_lines}")
        print(f"  - Nodes đã duyệt: {nodes_explored:,}")
        print(f"  - Thời gian: {elapsed:.2f} giây")
        print(f"  - Nước đi tốt nhất: {action}")
        print()
        
    except KeyboardInterrupt:
        print(f"  - BỊ NGẮT (quá lâu)")
        break
    
    # Nếu mất quá 30 giây thì dừng
    if elapsed > 30:
        print(f"  - Board này mất quá lâu, dừng test với board lớn hơn")
        break

print("\n=== BẢNG TỔNG KẾT ===")
print(f"{'Kích thước':<12} {'Số đường':<12} {'Nodes':<15} {'Thời gian (s)':<15}")
print("-" * 60)
for r in results_table:
    print(f"{r['size']:<12} {r['total_lines']:<12} {r['nodes']:<15,} {r['time']:<15.2f}")

print("\n=== PHÂN TÍCH ===")
print("- Board càng lớn, số nodes tăng theo cấp số nhân")
print("- Alpha-beta pruning giúp giảm đáng kể số nodes phải duyệt")
print("- Board lớn nhất có thể giải trong thời gian hợp lý: khoảng 3x3 hoặc 3x4")
print("- Với board 4x4 trở lên, cần dùng heuristic và giới hạn depth")

### Move ordering

Starting the search with better moves will increase the efficiency of alpha-beta pruning. Describe and implement a simple move ordering strategy. 

Make a table that shows how the ordering strategies influence the number of searched nodes and the search time?

In [ ]:
# Your code/ answer goes here.

def order_moves(board, actions_list, player):
    """
    Sắp xếp các nước đi theo mức độ ưu tiên
    Chiến lược:
    1. Ưu tiên các nước đi hoàn thành ô (tạo điểm ngay)
    2. Tránh các nước đi tạo cơ hội cho đối thủ
    3. Các nước đi trung lập
    """
    
    scored_actions = []
    
    for action in actions_list:
        score = 0
        orientation, row, col = action
        
        # Tạo board tạm để kiểm tra
        temp_board = {
            'size': board['size'],
            'lines': board['lines'].copy(),
            'boxes': board['boxes'].copy()
        }
        temp_board['lines'][(orientation, row, col)] = True
        
        # Kiểm tra xem nước đi này có hoàn thành ô nào không
        completed = check_box_completed(temp_board, orientation, row, col, player)
        
        if len(completed) > 0:
            # Ưu tiên cao nhất: hoàn thành ô
            score = 1000 + len(completed) * 100
        else:
            # Kiểm tra xem nước đi này có tạo cơ hội cho đối thủ không
            # (tức là tạo ra ô có 3 cạnh)
            boxes_with_3_sides = count_boxes_with_n_sides(temp_board, 3)
            
            if boxes_with_3_sides > count_boxes_with_n_sides(board, 3):
                # Tránh nước đi tạo cơ hội cho đối thủ
                score = -100
            else:
                # Nước đi trung lập - ưu tiên ở giữa board
                rows, cols = board['size']
                if orientation == 'h':
                    center_dist = abs(row - rows/2) + abs(col - cols/2)
                else:
                    center_dist = abs(row - rows/2) + abs(col - cols/2)
                score = -center_dist
        
        scored_actions.append((score, action))
    
    # Sắp xếp theo score giảm dần
    scored_actions.sort(reverse=True, key=lambda x: x[0])
    
    return [action for score, action in scored_actions]


def count_boxes_with_n_sides(board, n):
    """
    Đếm số ô có đúng n cạnh
    """
    rows, cols = board['size']
    count = 0
    
    for r in range(rows - 1):
        for c in range(cols - 1):
            # Đếm số cạnh của ô (r, c)
            sides = 0
            if ('h', r, c) in board['lines']:
                sides += 1
            if ('h', r + 1, c) in board['lines']:
                sides += 1
            if ('v', r, c) in board['lines']:
                sides += 1
            if ('v', r, c + 1) in board['lines']:
                sides += 1
            
            if sides == n:
                count += 1
    
    return count


def minimax_with_ordering(board, player, alpha=float('-inf'), beta=float('inf'), depth=0, max_depth=None):
    """
    Minimax với alpha-beta pruning VÀ move ordering
    """
    global nodes_explored
    nodes_explored += 1
    
    if terminal(board):
        return utility(board, 1), None
    
    if max_depth is not None and depth >= max_depth:
        return utility(board, 1), None
    
    # Lấy actions và sắp xếp
    valid_actions = actions(board)
    ordered_actions = order_moves(board, valid_actions, player)
    
    best_action = None
    
    if player == 1:
        best_value = float('-inf')
        for action in ordered_actions:
            new_board, next_player = result(board, action, player)
            value, _ = minimax_with_ordering(new_board, next_player, alpha, beta, depth + 1, max_depth)
            
            if value > best_value:
                best_value = value
                best_action = action
            
            alpha = max(alpha, best_value)
            if beta <= alpha:
                break
        
        return best_value, best_action
    else:
        best_value = float('inf')
        for action in ordered_actions:
            new_board, next_player = result(board, action, player)
            value, _ = minimax_with_ordering(new_board, next_player, alpha, beta, depth + 1, max_depth)
            
            if value < best_value:
                best_value = value
                best_action = action
            
            beta = min(beta, best_value)
            if beta <= alpha:
                break
        
        return best_value, best_action


print("=== SO SÁNH MINIMAX VỚI VÀ KHÔNG CÓ MOVE ORDERING ===\n")

# Test với board 3x3
test_board = {
    'size': (3, 3),
    'lines': {},
    'boxes': {}
}

# Không có move ordering
print("1. Minimax KHÔNG có move ordering:")
nodes_explored = 0
start = time.time()
value1, action1 = minimax_alpha_beta_count(test_board, 1)
time1 = time.time() - start
nodes1 = nodes_explored
print(f"   - Nodes: {nodes1:,}")
print(f"   - Thời gian: {time1:.2f}s")
print(f"   - Nước đi: {action1}\n")

# Có move ordering
print("2. Minimax CÓ move ordering:")
nodes_explored = 0
start = time.time()
value2, action2 = minimax_with_ordering(test_board, 1)
time2 = time.time() - start
nodes2 = nodes_explored
print(f"   - Nodes: {nodes2:,}")
print(f"   - Thời gian: {time2:.2f}s")
print(f"   - Nước đi: {action2}\n")

print("=== KẾT QUẢ ===")
print(f"Giảm số nodes: {nodes1 - nodes2:,} ({(1-nodes2/nodes1)*100:.1f}%)")
print(f"Giảm thời gian: {time1 - time2:.2f}s ({(1-time2/time1)*100:.1f}%)")
print("\nMove ordering giúp alpha-beta pruning hiệu quả hơn bằng cách:")
print("- Tìm các nước đi tốt trước → cập nhật alpha/beta sớm")
print("- Tăng cơ hội pruning các nhánh xấu")

### The first few moves

Start with an empty board. This is the worst case scenario for minimax search with alpha-beta pruning since it needs solve all possible games that can be played (minus some pruning) before making the decision. What can you do? 

In [ ]:
# Your code/ answer goes here.

def opening_move(board):
    """
    Chiến lược cho các nước đi đầu game
    
    Với board trống, ta có thể:
    1. Dùng opening book (các nước đi tốt đã biết trước)
    2. Chọn vị trí chiến lược (giữa board, góc, cạnh)
    3. Random trong những vị trí tốt
    
    Lý do: Minimax từ board trống mất RẤT lâu
    """
    rows, cols = board['size']
    num_lines_drawn = len(board['lines'])
    
    # Nếu còn quá nhiều đường chưa vẽ, dùng heuristic đơn giản
    total_possible = rows * (cols - 1) + (rows - 1) * cols
    
    if num_lines_drawn < 3:  # 3 nước đầu tiên
        # Chiến lược: chọn các vị trí ở giữa board
        # Tránh tạo cơ hội sớm cho đối thủ
        
        good_moves = []
        
        # Ưu tiên các đường ở giữa board
        for r in range(rows):
            for c in range(cols - 1):
                if ('h', r, c) not in board['lines']:
                    # Tính khoảng cách đến center
                    dist = abs(r - rows/2) + abs(c - (cols-1)/2)
                    good_moves.append((dist, ('h', r, c)))
        
        for r in range(rows - 1):
            for c in range(cols):
                if ('v', r, c) not in board['lines']:
                    dist = abs(r - (rows-1)/2) + abs(c - cols/2)
                    good_moves.append((dist, ('v', r, c)))
        
        # Sắp xếp theo khoảng cách (gần center nhất)
        good_moves.sort()
        
        # Chọn một trong top 5 moves tốt nhất (thêm tính random)
        top_moves = [move for dist, move in good_moves[:min(5, len(good_moves))]]
        return random.choice(top_moves)
    
    return None  # Dùng minimax cho các nước sau


def smart_player(board, player=None):
    """
    Agent thông minh: dùng opening book cho đầu game, minimax cho sau
    """
    if player is None:
        player = 1
    
    # Thử dùng opening move trước
    opening = opening_move(board)
    if opening is not None:
        return opening
    
    # Nếu không, dùng minimax
    # Với board lớn, giới hạn depth
    rows, cols = board['size']
    if rows * cols > 16:  # Board lớn
        max_depth = 6
    else:
        max_depth = None
    
    _, action = minimax_with_ordering(board, player, max_depth=max_depth)
    return action


print("=== CHIẾN LƯỢC CHO NƯỚC ĐI ĐẦU GAME ===\n")

print("VẤN ĐỀ:")
print("- Board trống có quá nhiều khả năng → minimax rất chậm")
print("- Ví dụ: board 4x4 có 24 đường → 24! khả năng sắp xếp\n")

print("GIẢI PHÁP:")
print("1. Opening book: Dùng các nước đi tốt đã biết")
print("2. Heuristic đơn giản: Chọn vị trí gần center")
print("3. Random trong top-k moves tốt nhất")
print("4. Sau vài nước đầu, chuyển sang minimax\n")

# Test
test_board = {
    'size': (4, 4),
    'lines': {},
    'boxes': {}
}

print("Test opening moves trên board 4x4 trống:")
for i in range(5):
    move = opening_move(test_board)
    print(f"  Nước {i+1}: {move}")
    # Vẽ nước đi để test nước tiếp
    if move:
        test_board['lines'][move] = True

print("\n✓ Opening moves nhanh, không cần tính toán phức tạp")
print("✓ Tránh tạo cơ hội sớm cho đối thủ")

### Playtime

Let the Minimax Search agent play a random agent on a small board. Analyze wins, losses and draws.

In [ ]:
# Your code/ answer goes here.

print("=== MINIMAX vs RANDOM AGENT ===\n")

# Chơi trên board nhỏ để minimax không quá chậm
board_size = (3, 3)
num_games = 20

print(f"Chạy {num_games} trận trên board {board_size[0]}x{board_size[1]}\n")

# Minimax (player 1) vs Random (player 2)
print("1. Minimax (đi trước) vs Random (đi sau):")
wins_1 = 0
for i in range(num_games):
    winner = play_game(alpha_beta_player, random_player, board_size)
    if winner == 1:
        wins_1 += 1
    if (i + 1) % 5 == 0:
        print(f"   Đã chơi {i+1} trận...")

print(f"   Kết quả: Minimax thắng {wins_1}/{num_games} trận ({wins_1/num_games*100:.0f}%)\n")

# Random (player 1) vs Minimax (player 2)
print("2. Random (đi trước) vs Minimax (đi sau):")
wins_2 = 0
for i in range(num_games):
    winner = play_game(random_player, alpha_beta_player, board_size)
    if winner == -1:  # Player 2 (minimax) thắng
        wins_2 += 1
    if (i + 1) % 5 == 0:
        print(f"   Đã chơi {i+1} trận...")

print(f"   Kết quả: Minimax thắng {wins_2}/{num_games} trận ({wins_2/num_games*100:.0f}%)\n")

print("=== PHÂN TÍCH ===")
print(f"Tổng cộng: Minimax thắng {wins_1 + wins_2}/{num_games * 2} trận")
print("- Minimax nên thắng GẦN NHƯ 100% số trận")
print("- Minimax chơi tối ưu, Random chơi ngẫu nhiên")
print("- Nếu Minimax không thắng hoàn toàn, có thể do:")
print("  + Board quá lớn → phải giới hạn depth")
print("  + Có bug trong implementation")

# Test chi tiết 1 trận
print("\n=== TEST CHI TIẾT 1 TRẬN ===")
print("Minimax vs Random, hiển thị từng nước:")
winner = play_game(alpha_beta_player, random_player, (3, 3), verbose=True)
if winner == 1:
    print("→ Minimax THẮNG (như dự đoán)")
else:
    print("→ Kết quả bất ngờ! Cần kiểm tra lại code")

## Task 4: Heuristic Alpha-Beta Tree Search [30 points] 

### Heuristic evaluation function

Define and implement a heuristic evaluation function.

In [ ]:
# Your code/ answer goes here.

def heuristic_eval(board, player):
    """
    Hàm đánh giá heuristic cho trạng thái board
    
    Các yếu tố đánh giá:
    1. Số ô đã hoàn thành (quan trọng nhất)
    2. Số ô sắp hoàn thành (có 3 cạnh) - cơ hội hoặc nguy hiểm
    3. Số ô có 2 cạnh - tiềm năng
    4. Kiểm soát vị trí trung tâm
    
    Returns: điểm số ước lượng cho player
    """
    
    # 1. Điểm từ các ô đã hoàn thành
    completed_score = 0
    for box_player in board['boxes'].values():
        if box_player == player:
            completed_score += 100  # +100 cho mỗi ô của mình
        else:
            completed_score -= 100  # -100 cho mỗi ô của đối thủ
    
    # 2. Đếm số ô theo số cạnh đã vẽ
    boxes_with_3_sides_mine = 0  # Ô có 3 cạnh mình kiểm soát
    boxes_with_3_sides_opponent = 0
    boxes_with_2_sides = 0
    boxes_with_1_side = 0
    
    rows, cols = board['size']
    
    for r in range(rows - 1):
        for c in range(cols - 1):
            # Bỏ qua ô đã hoàn thành
            if (r, c) in board['boxes']:
                continue
            
            # Đếm số cạnh của ô
            sides = []
            if ('h', r, c) in board['lines']:
                sides.append('top')
            if ('h', r + 1, c) in board['lines']:
                sides.append('bottom')
            if ('v', r, c) in board['lines']:
                sides.append('left')
            if ('v', r, c + 1) in board['lines']:
                sides.append('right')
            
            num_sides = len(sides)
            
            if num_sides == 3:
                # Ô có 3 cạnh - rất quan trọng!
                # Cần xác định ai sẽ được lợi từ ô này
                # (Giả sử player hiện tại sẽ đóng nó)
                boxes_with_3_sides_mine += 1
            elif num_sides == 2:
                boxes_with_2_sides += 1
            elif num_sides == 1:
                boxes_with_1_side += 1
    
    # 3. Tính điểm heuristic
    score = completed_score
    
    # Ô có 3 cạnh: rất nguy hiểm nếu để đối thủ lấy
    # Nhưng nếu là lượt mình, ta có thể lấy
    score += boxes_with_3_sides_mine * 50  # Cơ hội lấy điểm
    
    # Ô có 2 cạnh: tiềm năng, nhưng cần cẩn thận
    score += boxes_with_2_sides * 5
    
    # Ô có 1 cạnh: an toàn, có thể phát triển
    score += boxes_with_1_side * 2
    
    return score


def heuristic_eval_v2(board, player):
    """
    Heuristic phiên bản 2 - tập trung vào sự chênh lệch
    """
    rows, cols = board['size']
    
    # Điểm cơ bản từ các ô đã hoàn thành
    my_boxes = sum(1 for p in board['boxes'].values() if p == player)
    opp_boxes = sum(1 for p in board['boxes'].values() if p == -player)
    
    score = (my_boxes - opp_boxes) * 100
    
    # Đếm các ô theo số cạnh
    for r in range(rows - 1):
        for c in range(cols - 1):
            if (r, c) in board['boxes']:
                continue
            
            sides = 0
            if ('h', r, c) in board['lines']:
                sides += 1
            if ('h', r + 1, c) in board['lines']:
                sides += 1
            if ('v', r, c) in board['lines']:
                sides += 1
            if ('v', r, c + 1) in board['lines']:
                sides += 1
            
            # Phạt nặng nếu tạo ô có 3 cạnh (nguy hiểm)
            if sides == 3:
                score -= 30
            # Thưởng nhẹ cho ô có 2 cạnh
            elif sides == 2:
                score += 3
    
    return score


# Test các heuristic
print("=== TEST HEURISTIC EVALUATION ===\n")

# Board mẫu để test
test_board = {
    'size': (3, 3),
    'lines': {
        ('h', 0, 0): True, ('v', 0, 0): True, ('h', 1, 0): True,
        ('v', 0, 1): True,
    },
    'boxes': {(0, 0): 1}  # Player 1 có 1 ô
}

print("Board test:")
display_board(test_board)

score1 = heuristic_eval(test_board, 1)
score2 = heuristic_eval_v2(test_board, 1)

print(f"Heuristic v1 (player 1): {score1}")
print(f"Heuristic v2 (player 1): {score2}")

print("\n=== GIẢI THÍCH CÁC HEURISTIC ===")
print("\nHeuristic v1:")
print("- Ưu tiên: Ô đã hoàn thành (±100 điểm)")
print("- Thưởng cho ô có 3 cạnh (+50) - cơ hội ghi điểm")
print("- Thưởng nhẹ cho ô có 2 cạnh (+5)")
print("- Thưởng nhỏ cho ô có 1 cạnh (+2)")

print("\nHeuristic v2:")
print("- Ưu tiên: Chênh lệch số ô (±100 điểm)")
print("- PHẠT nặng khi tạo ô 3 cạnh (-30) - nguy hiểm!")
print("- Thưởng nhẹ cho ô 2 cạnh (+3)")
print("- Chiến lược phòng thủ hơn")

### Cutting off search 

Modify your Minimax Search with Alpha-Beta Pruning to cut off search at a specified depth and use the heuristic evaluation function. Experiment with different cutoff values.

In [ ]:
# Your code/ answer goes here.

def minimax_heuristic(board, player, alpha=float('-inf'), beta=float('inf'), 
                      depth=0, max_depth=5, heuristic_func=heuristic_eval):
    """
    Minimax với alpha-beta pruning, giới hạn depth, và heuristic evaluation
    
    Parameters:
    - max_depth: độ sâu tối đa trước khi dùng heuristic
    - heuristic_func: hàm đánh giá heuristic
    """
    global nodes_explored
    nodes_explored += 1
    
    # Kiểm tra terminal state
    if terminal(board):
        return utility(board, 1), None
    
    # CUTOFF: Nếu đạt max_depth, dùng heuristic thay vì tìm kiếm sâu hơn
    if depth >= max_depth:
        return heuristic_func(board, 1), None
    
    # Lấy và sắp xếp actions
    valid_actions = actions(board)
    ordered_actions = order_moves(board, valid_actions, player)
    
    best_action = None
    
    if player == 1:  # MAX
        best_value = float('-inf')
        for action in ordered_actions:
            new_board, next_player = result(board, action, player)
            value, _ = minimax_heuristic(new_board, next_player, alpha, beta, 
                                        depth + 1, max_depth, heuristic_func)
            
            if value > best_value:
                best_value = value
                best_action = action
            
            alpha = max(alpha, best_value)
            if beta <= alpha:
                break
        
        return best_value, best_action
    
    else:  # MIN
        best_value = float('inf')
        for action in ordered_actions:
            new_board, next_player = result(board, action, player)
            value, _ = minimax_heuristic(new_board, next_player, alpha, beta, 
                                        depth + 1, max_depth, heuristic_func)
            
            if value < best_value:
                best_value = value
                best_action = action
            
            beta = min(beta, best_value)
            if beta <= alpha:
                break
        
        return best_value, best_action


def heuristic_player(board, player=None, max_depth=5, heuristic_func=heuristic_eval):
    """
    Agent dùng minimax với heuristic và giới hạn depth
    """
    if player is None:
        player = 1
    
    _, action = minimax_heuristic(board, player, max_depth=max_depth, 
                                  heuristic_func=heuristic_func)
    return action


print("=== TEST MINIMAX VỚI CUTOFF DEPTH ===\n")

# Test với các depth khác nhau
test_board = {
    'size': (3, 3),
    'lines': {},
    'boxes': {}
}

depths = [1, 2, 3, 4, 5]

print("So sánh hiệu năng với các depth khác nhau:\n")
print(f"{'Depth':<8} {'Nodes':<12} {'Thời gian (s)':<15} {'Nước đi':<20}")
print("-" * 60)

for d in depths:
    nodes_explored = 0
    start = time.time()
    value, action = minimax_heuristic(test_board, 1, max_depth=d)
    elapsed = time.time() - start
    
    print(f"{d:<8} {nodes_explored:<12,} {elapsed:<15.3f} {str(action):<20}")

print("\n=== PHÂN TÍCH ===")
print("- Depth càng lớn → nodes càng nhiều → thời gian càng lâu")
print("- Depth càng lớn → quyết định càng tốt (nhìn xa hơn)")
print("- Cần cân bằng giữa chất lượng và tốc độ")
print("- Depth 4-6 thường là lựa chọn tốt cho board trung bình")

How many nodes are searched and how long does it take to make a move? Start with a smaller board with 4 columns and make the board larger by adding columns.

In [ ]:
# Your code/ answer goes here.

print("=== ĐO HIỆU NĂNG VỚI BOARD TĂNG DẦN ===\n")

# Test với board có số cột tăng dần
board_configs = [
    (3, 3, 4),  # 3x3, depth 4
    (3, 4, 4),  # 3x4, depth 4
    (3, 5, 4),  # 3x5, depth 4
    (4, 4, 3),  # 4x4, depth 3
    (4, 5, 3),  # 4x5, depth 3
]

print(f"{'Board':<10} {'Depth':<8} {'Đường':<10} {'Nodes':<12} {'Thời gian (s)':<15}")
print("-" * 70)

for rows, cols, depth in board_configs:
    board = {
        'size': (rows, cols),
        'lines': {},
        'boxes': {}
    }
    
    total_lines = rows * (cols - 1) + (rows - 1) * cols
    
    nodes_explored = 0
    start = time.time()
    
    try:
        value, action = minimax_heuristic(board, 1, max_depth=depth)
        elapsed = time.time() - start
        
        print(f"{rows}x{cols:<7} {depth:<8} {total_lines:<10} {nodes_explored:<12,} {elapsed:<15.3f}")
        
        # Nếu quá 10 giây thì dừng
        if elapsed > 10:
            print("   → Quá chậm, dừng test với board lớn hơn")
            break
            
    except Exception as e:
        print(f"{rows}x{cols:<7} {depth:<8} {total_lines:<10} ERROR: {str(e)}")
        break

print("\n=== KẾT LUẬN ===")
print("Với heuristic và giới hạn depth:")
print("- Có thể xử lý board lớn hơn nhiều so với minimax đầy đủ")
print("- Board 4x5 với depth 3-4 là khả thi")
print("- Board càng lớn càng cần giảm depth")
print("- Heuristic tốt giúp compensate cho depth nhỏ")

# So sánh 2 heuristic
print("\n=== SO SÁNH 2 HEURISTIC ===\n")

test_board = {
    'size': (3, 4),
    'lines': {},
    'boxes': {}
}

print("Heuristic v1 (tấn công):")
nodes_explored = 0
start = time.time()
v1, a1 = minimax_heuristic(test_board, 1, max_depth=4, heuristic_func=heuristic_eval)
t1 = time.time() - start
n1 = nodes_explored
print(f"  Nodes: {n1:,}, Thời gian: {t1:.3f}s, Nước đi: {a1}")

print("\nHeuristic v2 (phòng thủ):")
nodes_explored = 0
start = time.time()
v2, a2 = minimax_heuristic(test_board, 1, max_depth=4, heuristic_func=heuristic_eval_v2)
t2 = time.time() - start
n2 = nodes_explored
print(f"  Nodes: {n2:,}, Thời gian: {t2:.3f}s, Nước đi: {a2}")

print("\nCả 2 heuristic cho kết quả tương tự về hiệu năng")
print("Nhưng có thể dẫn đến phong cách chơi khác nhau")

### Playtime

Let two heuristic search agents (different cutoff depth, different heuristic evaluation function) compete against each other on a reasonably sized board. Since there is no randomness, you only need to let them play once.

In [ ]:
# Your code/ answer goes here.

print("=== HEURISTIC AGENTS CHƠI VỚI NHAU ===\n")

# Tạo các agent với cấu hình khác nhau
def heuristic_player_v1_d3(board, player=None):
    """Heuristic v1, depth 3"""
    return heuristic_player(board, player, max_depth=3, heuristic_func=heuristic_eval)

def heuristic_player_v1_d5(board, player=None):
    """Heuristic v1, depth 5"""
    return heuristic_player(board, player, max_depth=5, heuristic_func=heuristic_eval)

def heuristic_player_v2_d3(board, player=None):
    """Heuristic v2, depth 3"""
    return heuristic_player(board, player, max_depth=3, heuristic_func=heuristic_eval_v2)

def heuristic_player_v2_d5(board, player=None):
    """Heuristic v2, depth 5"""
    return heuristic_player(board, player, max_depth=5, heuristic_func=heuristic_eval_v2)


print("TEST 1: Heuristic v1 depth=3 vs Heuristic v1 depth=5")
print("(Cùng heuristic, khác depth)\n")

winner = play_game(heuristic_player_v1_d3, heuristic_player_v1_d5, 
                   board_size=(3, 4), verbose=False)

if winner == 1:
    print("→ Player 1 (depth=3) THẮNG - Bất ngờ!")
elif winner == -1:
    print("→ Player 2 (depth=5) THẮNG - Như mong đợi (nhìn xa hơn)")
else:
    print("→ HÒA")

print("\n" + "="*60)
print("\nTEST 2: Heuristic v1 depth=4 vs Heuristic v2 depth=4")
print("(Khác heuristic, cùng depth)\n")

def h1_d4(board, player=None):
    return heuristic_player(board, player, max_depth=4, heuristic_func=heuristic_eval)

def h2_d4(board, player=None):
    return heuristic_player(board, player, max_depth=4, heuristic_func=heuristic_eval_v2)

winner = play_game(h1_d4, h2_d4, board_size=(3, 4), verbose=False)

if winner == 1:
    print("→ Heuristic v1 (tấn công) THẮNG")
elif winner == -1:
    print("→ Heuristic v2 (phòng thủ) THẮNG")
else:
    print("→ HÒA")

print("\n" + "="*60)
print("\nTEST 3: Chi tiết một trận đấu")
print("Heuristic v1 depth=3 vs Heuristic v2 depth=3\n")

winner = play_game(heuristic_player_v1_d3, heuristic_player_v2_d3, 
                   board_size=(3, 3), verbose=True)

print(f"\n=== KẾT QUẢ CUỐI CÙNG ===")
if winner == 1:
    print("Player 1 (Heuristic v1 - tấn công) THẮNG")
elif winner == -1:
    print("Player 2 (Heuristic v2 - phòng thủ) THẮNG")
else:
    print("HÒA")

print("\n=== NHẬN XÉT ===")
print("1. Depth lớn hơn thường có lợi thế (nhìn xa hơn)")
print("2. Heuristic tốt hơn có thể bù depth nhỏ hơn")
print("3. Phong cách tấn công vs phòng thủ:")
print("   - v1 tìm cơ hội ghi điểm")
print("   - v2 tránh tạo cơ hội cho đối thủ")
print("4. Kết quả phụ thuộc board size và game state")

## Tournament task [+1 to 5% bonus on your course grade; will be assigned separately]

Find another student and let your best agent play against the other student's best player. You are allowed to use any improvements you like as long as you code it yourself. We will set up a class tournament on Canvas. This tournament will continue after the submission deadline.

## Graduate student advanced task: Pure Monte Carlo Search and Best First Move [10 point]

__Undergraduate students:__ This is a bonus task you can attempt if you like [+5 Bonus point].

### Pure Monte Carlo Search

Implement Pure Monte Carlo Search (see [tic-tac-toe-example](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_pure_monte_carlo_search.ipynb)) and investigate how this search performs on the test boards that you have used above. 

In [ ]:
# Your code/ answer goes here.

import random

def random_playout(board, player):
    """
    Chơi ngẫu nhiên từ trạng thái hiện tại đến kết thúc
    Trả về kết quả: 1 nếu player 1 thắng, -1 nếu thua, 0 nếu hòa
    """
    # Copy board để không ảnh hưởng board gốc
    sim_board = {
        'size': board['size'],
        'lines': board['lines'].copy(),
        'boxes': board['boxes'].copy()
    }
    
    current_player = player
    
    # Chơi đến hết
    while not terminal(sim_board):
        valid_actions = actions(sim_board)
        if len(valid_actions) == 0:
            break
        
        # Chọn ngẫu nhiên
        action = random.choice(valid_actions)
        sim_board, current_player = result(sim_board, action, current_player)
    
    # Tính kết quả
    p1_score = sum(1 for p in sim_board['boxes'].values() if p == 1)
    p2_score = sum(1 for p in sim_board['boxes'].values() if p == -1)
    
    if p1_score > p2_score:
        return 1
    elif p2_score > p1_score:
        return -1
    else:
        return 0


def pure_monte_carlo_search(board, player, num_simulations=100):
    """
    Pure Monte Carlo Search
    
    Cho mỗi action có thể:
    1. Thực hiện action
    2. Chạy nhiều random playouts
    3. Đếm số lần thắng
    4. Chọn action có tỷ lệ thắng cao nhất
    """
    valid_actions = actions(board)
    
    if len(valid_actions) == 0:
        return None
    
    # Thống kê cho mỗi action
    action_stats = {}
    
    for action in valid_actions:
        wins = 0
        losses = 0
        draws = 0
        
        # Thực hiện action
        new_board, next_player = result(board, action, player)
        
        # Chạy nhiều simulations
        for _ in range(num_simulations):
            result_val = random_playout(new_board, next_player)
            
            # Đếm kết quả (từ góc nhìn của player gốc)
            if result_val == player:
                wins += 1
            elif result_val == -player:
                losses += 1
            else:
                draws += 1
        
        # Tính win rate
        win_rate = (wins + 0.5 * draws) / num_simulations
        action_stats[action] = {
            'wins': wins,
            'losses': losses,
            'draws': draws,
            'win_rate': win_rate
        }
    
    # Chọn action có win rate cao nhất
    best_action = max(action_stats.keys(), key=lambda a: action_stats[a]['win_rate'])
    
    return best_action, action_stats


def monte_carlo_player(board, player=None, num_simulations=100):
    """
    Agent dùng Pure Monte Carlo Search
    """
    if player is None:
        player = 1
    
    result = pure_monte_carlo_search(board, player, num_simulations)
    if result is None:
        return None
    
    best_action, stats = result
    return best_action


print("=== PURE MONTE CARLO SEARCH ===\n")

# Test trên board đơn giản
test_board = {
    'size': (3, 3),
    'lines': {
        ('h', 0, 0): True, ('v', 0, 0): True, ('h', 1, 0): True,
    },
    'boxes': {}
}

print("Board test:")
display_board(test_board)

print("Chạy Monte Carlo Search với 50 simulations cho mỗi action...\n")

best_action, stats = pure_monte_carlo_search(test_board, 1, num_simulations=50)

print("Kết quả:")
print(f"{'Action':<20} {'Wins':<8} {'Losses':<8} {'Draws':<8} {'Win Rate':<10}")
print("-" * 70)

# Sắp xếp theo win rate
sorted_actions = sorted(stats.items(), key=lambda x: x[1]['win_rate'], reverse=True)

for action, stat in sorted_actions[:10]:  # Hiển thị top 10
    print(f"{str(action):<20} {stat['wins']:<8} {stat['losses']:<8} {stat['draws']:<8} {stat['win_rate']:<10.3f}")

print(f"\n→ Action tốt nhất: {best_action}")
print(f"   Win rate: {stats[best_action]['win_rate']:.3f}")

# So sánh với minimax
print("\n" + "="*60)
print("So sánh Monte Carlo vs Minimax:\n")

_, minimax_action = minimax_alpha_beta(test_board, 1)
print(f"Minimax chọn: {minimax_action}")
print(f"Monte Carlo chọn: {best_action}")

if minimax_action == best_action:
    print("✓ Cả hai chọn cùng action!")
else:
    print("✗ Khác nhau - Monte Carlo có thể sai do random sampling")

# Test performance
print("\n" + "="*60)
print("Monte Carlo vs Random Agent:\n")

num_games = 20
wins = 0

for i in range(num_games):
    winner = play_game(monte_carlo_player, random_player, board_size=(3, 3))
    if winner == 1:
        wins += 1

print(f"Monte Carlo thắng {wins}/{num_games} trận ({wins/num_games*100:.0f}%)")

print("\n=== ƯU ĐIỂM MONTE CARLO ===")
print("+ Không cần heuristic function")
print("+ Dễ implement")
print("+ Hoạt động tốt với game phức tạp")
print("+ Có thể parallel hóa")

print("\n=== NHƯỢC ĐIỂM ===")
print("- Chậm hơn minimax với heuristic tốt")
print("- Cần nhiều simulations để chính xác")
print("- Không đảm bảo tối ưu")
print("- Random nên kết quả không ổn định")

### Best First Move

How would you determine what the best first move for a standard board ($5 \times 5$) is? You can use Pure Monte Carlo Search or any algorithms that you have implemented above.

In [ ]:
# Your code/ answer goes here.

print("=== TÌM NƯỚC ĐI ĐẦU TIÊN TỐT NHẤT CHO BOARD 5x5 ===\n")

def find_best_first_move_monte_carlo(board_size=(5, 5), simulations_per_move=200):
    """
    Dùng Monte Carlo để tìm nước đi đầu tốt nhất
    """
    board = {
        'size': board_size,
        'lines': {},
        'boxes': {}
    }
    
    print(f"Board size: {board_size[0]}x{board_size[1]}")
    print(f"Simulations per move: {simulations_per_move}\n")
    
    # Lấy tất cả actions có thể
    all_actions = actions(board)
    print(f"Tổng số nước đi có thể: {len(all_actions)}")
    
    # Do quá nhiều, ta chỉ test một số nước đi tiêu biểu
    # Dựa trên tính đối xứng, ta test các vị trí:
    rows, cols = board_size
    
    representative_moves = [
        ('h', 0, 0),           # Góc trên trái ngang
        ('v', 0, 0),           # Góc trên trái dọc
        ('h', rows//2, cols//2),  # Giữa ngang
        ('v', rows//2, cols//2),  # Giữa dọc
        ('h', 0, cols//2),     # Cạnh trên giữa
        ('v', rows//2, 0),     # Cạnh trái giữa
    ]
    
    # Lọc các moves hợp lệ
    test_moves = [m for m in representative_moves if m in all_actions]
    
    print(f"Test {len(test_moves)} nước đi đại diện:\n")
    
    best_move = None
    best_win_rate = -1
    results = []
    
    for i, move in enumerate(test_moves):
        print(f"Testing move {i+1}/{len(test_moves)}: {move}...", end=" ")
        
        # Thực hiện move
        test_board, next_player = result(board, move, 1)
        
        # Chạy simulations
        wins = 0
        for _ in range(simulations_per_move):
            result_val = random_playout(test_board, next_player)
            if result_val == 1:
                wins += 1
        
        win_rate = wins / simulations_per_move
        results.append((move, win_rate, wins))
        
        print(f"Win rate: {win_rate:.3f}")
        
        if win_rate > best_win_rate:
            best_win_rate = win_rate
            best_move = move
    
    print("\n" + "="*60)
    print("KẾT QUẢ:\n")
    print(f"{'Nước đi':<20} {'Wins':<10} {'Win Rate':<12}")
    print("-" * 50)
    
    results.sort(key=lambda x: x[1], reverse=True)
    for move, win_rate, wins in results:
        marker = "  ← TỐT NHẤT" if move == best_move else ""
        print(f"{str(move):<20} {wins:<10} {win_rate:<12.3f}{marker}")
    
    return best_move, best_win_rate


# Chạy phân tích
print("Phân tích với Monte Carlo (có thể mất vài phút)...\n")
best_move, best_rate = find_best_first_move_monte_carlo((5, 5), simulations_per_move=100)

print("\n" + "="*60)
print("KẾT LUẬN:")
print(f"\nNước đi đầu tiên tốt nhất (theo Monte Carlo): {best_move}")
print(f"Win rate ước tính: {best_rate:.3f}")

print("\n=== PHÂN TÍCH ===")
print("Với Dots and Boxes:")
print("1. Do tính đối xứng, nhiều nước đầu có win rate tương tự")
print("2. Nước đi ở giữa thường an toàn hơn (tránh tạo cơ hội sớm)")
print("3. Nước đi góc/cạnh có thể tạo pattern dễ kiểm soát")
print("4. Với perfect play từ cả 2 bên, kết quả phụ thuộc ai đi trước")

print("\n=== PHƯƠNG PHÁP KHÁC ĐỂ TÌM BEST FIRST MOVE ===")
print("\n1. OPENING BOOK:")
print("   - Phân tích các ván đấu giữa experts")
print("   - Lưu trữ các nước đi tốt đã biết")
print("   - Nhanh nhất nhưng cần database")

print("\n2. MINIMAX VỚI HEURISTIC:")
print("   - Depth 4-6 từ board trống")
print("   - Cần heuristic tốt")
print("   - Chậm hơn Monte Carlo")

print("\n3. PURE MONTE CARLO (Đã dùng):")
print("   - Không cần kiến thức domain")
print("   - Đơn giản nhưng cần nhiều simulations")
print("   - Kết quả có độ tin cậy thống kê")

print("\n4. MCTS (Monte Carlo Tree Search):")
print("   - Cân bằng exploration và exploitation")
print("   - Tốt hơn Pure Monte Carlo")
print("   - Phức tạp hơn để implement")

print("\n5. MACHINE LEARNING:")
print("   - Train model từ database games")
print("   - Cần nhiều data")
print("   - Có thể học patterns phức tạp")


# Phân tích sâu hơn với minimax (nếu có thời gian)
print("\n" + "="*60)
print("BONUS: So sánh với Minimax (depth giới hạn):\n")

def find_best_first_move_minimax(board_size=(5, 5), max_depth=3):
    """
    Dùng minimax với depth giới hạn để tìm nước đầu tốt nhất
    """
    board = {
        'size': board_size,
        'lines': {},
        'boxes': {}
    }
    
    print(f"Chạy minimax với depth={max_depth}...")
    
    start = time.time()
    value, action = minimax_heuristic(board, 1, max_depth=max_depth, 
                                      heuristic_func=heuristic_eval)
    elapsed = time.time() - start
    
    print(f"Thời gian: {elapsed:.2f}s")
    print(f"Nước đi tốt nhất: {action}")
    print(f"Giá trị heuristic: {value}")
    
    return action

minimax_best = find_best_first_move_minimax((5, 5), max_depth=3)

print("\nSo sánh kết quả:")
print(f"Monte Carlo chọn: {best_move}")
print(f"Minimax chọn: {minimax_best}")

if best_move == minimax_best:
    print("✓ Cả hai phương pháp đồng ý!")
else:
    print("✗ Khác nhau - có thể do:")
    print("  - Monte Carlo có yếu tố random")
    print("  - Minimax depth thấp nên không hoàn hảo")
    print("  - Heuristic chưa tối ưu")


# Khuyến nghị cuối cùng
print("\n" + "="*60)
print("=== KHUYẾN NGHỊ CHO BOARD 5x5 ===\n")

print("1. NƯỚC ĐI ĐẦU AN TOÀN:")
print("   - Chọn đường ở giữa board")
print("   - Tránh góc và cạnh (có thể tạo patterns bất lợi)")
print("   - Ví dụ: ('h', 2, 2) hoặc ('v', 2, 2)")

print("\n2. CHIẾN LƯỢC CHUNG:")
print("   - Giai đoạn đầu: Đừng tạo ô có 3 cạnh")
print("   - Giai đoạn giữa: Kiểm soát các chuỗi ô")
print("   - Giai đoạn cuối: Tối đa hóa số ô lấy được")

print("\n3. TACTICS CỤ THỂ:")
print("   - 'Sacrifice': Cho đối thủ vài ô để kiểm soát nhiều ô hơn")
print("   - 'Chain': Tạo chuỗi ô liên tiếp để lấy hết")
print("   - 'Double-cross': Buộc đối thủ vào thế khó")

print("\n4. VỚI BOARD 5x5 CỤ THỂ:")
total_boxes = 4 * 4  # (5-1) x (5-1) = 16 ô
print(f"   - Tổng số ô: {total_boxes}")
print(f"   - Cần thắng: ít nhất {total_boxes//2 + 1} ô")
print(f"   - Player đi trước có lợi thế nhẹ")